## Text Preprocessing + Intent Simulation
Create:  
- Tokenization
- Lemmatization
- Stopword removal
- Intent taxonomy creation
- Simulating intent labels
- Simulating confidence scores
- Defining fallback logic

### Import data


In [2]:
# Import data
import pandas as pd

df = pd.read_csv("../data/twcs_clean.csv")
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,Clean_text,conversation_id
0,192624,161253,True,Wed Oct 04 13:59:33 +0000 2017,@161252 What's that egg website people talk about,192623,192625.0,what s that egg website people talk about,192625.0
1,738238,296574,True,Fri Oct 06 18:29:06 +0000 2017,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...,738237,NaN,why,738238.0
2,2414302,AppleSupport,False,Tue Nov 14 17:38:01 +0000 2017,@693975 We can assist you. We recommend updati...,2414303,2414304.0,we can assist you we recommend updating to ios...,2414304.0
3,1793929,539096,True,Thu Oct 12 06:04:41 +0000 2017,@331912 @115955 Thats better than having an un...,1793928,1793930.0,thats better than having an unstable connectio...,1793930.0
4,2088018,617376,True,Mon Nov 06 20:30:49 +0000 2017,@VirginAmerica is probably one of the best air...,2088017,NaN,is probably one of the best airlines i ve ever...,2088018.0


## Step 2 — Load spaCy model

###  spaCy model is a pre‑built package of linguistic knowledge that spaCy uses to analyze text
Check doc/spaCy.md for more information

Note: ensure you have pip insalled spaCy in your venv

In [3]:
import spacy
nlp = spacy.load("en_core_web_sm")


## Step 3 — Lemmatize text
converts words to their base/original form called "Lemma":  
- “running” → “run”  
- “better” → “good”  
- “messages” → “message”  

This help create a normalized text column is a column in your dataset where the text has been cleaned, standardized, and transformed into a consistent format so it’s easier for models or rules to process.  

#### What “normalized text” usually includes
Normalization can involve one or more of these transformations:  
Lowercasing : “Hello WORLD” → “hello world”  

Removing punctuation : “Hi, Divya!” → “hi divya”  

Removing extra whitespace: “AI   agents” → “AI agents”

Expanding contractions: “don’t” → “do not”

Removing stopwords (optional): “this is a test” → “test”

Lemmatization: “running” → “run”

Unicode normalization: “café” → “cafe”

Removing special characters or HTML tags

### Check for missing values in clean text and fill it with empty string

In [4]:
df["Clean_text"].isna().sum()


np.int64(35)

In [5]:
df["Clean_text"] = df["Clean_text"].fillna("")
df["Clean_text"].isna().sum()

np.int64(0)

#### Convert data type to str

In [6]:
df["Clean_text"] = df["Clean_text"].astype(str)

### Create a lammatize function 

In [7]:
def preprocess_spacy(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop]
    return " ".join(tokens)

df["text_lemma"] = df["Clean_text"].apply(preprocess_spacy)
df[["Clean_text", "text_lemma"]].head()


,Clean_text,text_lemma
0,what s that egg website people talk about,s egg website people talk
1,why,
2,we can assist you we recommend updating to ios...,assist recommend update io 11 1 1 haven t chan...
3,thats better than having an unstable connectio...,s well have unstable connection drop 5 20 min
4,is probably one of the best airlines i ve ever...,probably good airline ve experience


### Explaination:  
1. s egg website people talk  
✔ “what” removed (stopword)  
✔ “that” removed (stopword)  
✔ “talking” → “talk” (lemmatized)  
✔ “egg website” preserved  

2. empty(why) :  
✔ “why” is a stopword → removed  
✔ Empty string is expected  

3. assist recommend update io 11 1 1 haven t chan...  
✔ “we”, “can”, “you” removed (stopwords)  
✔ “updating” → “update”  
✔ “haven’t” → “have” (lemmatized)  
✔ Numbers preserved  

### Create a simple intent taxonomy
We’ll start with 8 beginner-friendly intents:

In [8]:
intents = [
    "billing_issue",
    "technical_problem",
    "account_access",
    "order_status",
    "refund_request",
    "general_query",
    "smalltalk",
    "unknown"
]


### Step 5 — Simulate intent labels

**Note:** Since this dataset has no labels, we simulate them for now.  
Twitter customer support data is raw text. There is no ground truth for:  
intent, entity , confidence, fallback

So we simulate them to:
Build the pipeline, Learn the workflow, Practice analytics, Build dashboards, Understand taxonomy gaps

In [9]:
import numpy as np

df["intent"] = np.random.choice(intents, size=len(df))


### Step 6 — Simulate confidence scores

This mimics real NLU model behavior.

In [10]:
df["confidence"] = np.random.uniform(0.40, 0.99, size=len(df))


### Step 7 — Define fallback logic
Fallback happens when:  
Intent = unknown  
OR confidence < threshold

**Note:** Confidence scores are normally produced by:  
Rasa NLU , Dialogflow, LUIS, BERT-based classifiers, Proprietary enterprise NLU models

But since we don’t have a model yet, we simulate confidence to:  
Practice fallback logic, Practice low-confidence analysis, Build dashboards, Learn how to interpret NLU outputs

In [11]:
low_conf_threshold = 0.6

df["is_low_conf"] = df["confidence"] < low_conf_threshold
df["is_fallback"] = (df["intent"] == "unknown") | df["is_low_conf"]


### Classifying inbound for analytics
- inbound = True → customer message - `is_company`
- inbound = False → company response - `is_user`

In [20]:
# classify the inbound
df['is_user'] = (df['inbound'] == True) # or df["is_user"] = df["inbound"] 
df["is_company"] = ~df["inbound"] 


In [22]:
df[["inbound", "is_user", "is_company"]].head()


,inbound,is_user,is_company
0,True,True,False
1,True,True,False
2,False,False,True
3,True,True,False
4,True,True,False


### Step 8 — Save updated dataset

In [23]:
df.to_csv("../data/twcs_intent_simulated.csv", index=False)


In [24]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,Clean_text,conversation_id,text_lemma,intent,confidence,is_low_conf,is_fallback,is_user,is_company
0,192624,161253,True,Wed Oct 04 13:59:33 +0000 2017,@161252 What's that egg website people talk about,192623,192625.0,what s that egg website people talk about,192625.0,s egg website people talk,refund_request,0.631970,False,False,True,False
1,738238,296574,True,Fri Oct 06 18:29:06 +0000 2017,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...,738237,NaN,why,738238.0,,technical_problem,0.682013,False,False,True,False
2,2414302,AppleSupport,False,Tue Nov 14 17:38:01 +0000 2017,@693975 We can assist you. We recommend updati...,2414303,2414304.0,we can assist you we recommend updating to ios...,2414304.0,assist recommend update io 11 1 1 haven t chan...,account_access,0.891190,False,False,False,True
3,1793929,539096,True,Thu Oct 12 06:04:41 +0000 2017,@331912 @115955 Thats better than having an un...,1793928,1793930.0,thats better than having an unstable connectio...,1793930.0,s well have unstable connection drop 5 20 min,technical_problem,0.433034,True,True,True,False
4,2088018,617376,True,Mon Nov 06 20:30:49 +0000 2017,@VirginAmerica is probably one of the best air...,2088017,NaN,is probably one of the best airlines i ve ever...,2088018.0,probably good airline ve experience,refund_request,0.427244,True,True,True,False
